# Apply the data to the model and create "pickles/model_on_data.pkl"
----

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from get_model_probabilities import *
import scienceplots
plt.style.use(["science","grid"])

Source redshift:1.36


In [9]:
#RUN ALL MODELS ON THE TEST DATA
#-------------------------------
args.source_domain='a2744'
args.target_domain='a2744'
filter_list = ['concat','f115w','f150w']

models = {}
probabilities = {}
probabilities_noise = {}

for ifx, ifilter in enumerate(filter_list):
    
    
    cdan_models = np.sort(glob(
        f"../models/{ifilter}/cdan_baha2dark_pre_squeezenet1_aw_1_pad_shear_avgpool_gauss_seed_*_nob1_ft_tweaked_align_10_best.pth"
    ))
    all_models = []
    for imodel in cdan_models:
        args.checkpoint = imodel
        this_model = create_model(args)
        
        model_name = f"seed_{this_model.args.seed}"
        #if target_test_accuracy[ifilter][model_name]['target_test_f1'] <0.48:
        #    continue
            
        all_models.append(this_model)
    print(f"Found {len(all_models)} Models")
    models[ifilter] = all_models
    meta, data = pkl.load(open(f"../data/a2744/obs_data_{ifilter}.pkl","rb"))

    #repeat 
    probabilities_filt = []
    probabilities_noise_filt = []

    with torch.no_grad():

        for imodel in tqdm(all_models):
            outputs_dict = imodel([torch.tensor(data,dtype=torch.float32)])
            #if torch.all(torch.abs(outputs_dict['classification'][:,0] - outputs_dict['classification'][0,0])  < 0.01):
            #   continue
            probabilities_filt.append(torch.softmax( outputs_dict['classification'], dim=1 )[0,0]) 
            probabilities_noise_filt.append(torch.softmax( outputs_dict['classification'], dim=1 )[1:,0])
            

    probabilities_filt = torch.tensor( probabilities_filt).detach().numpy()
    probabilities_noise_filt =  torch.stack( probabilities_noise_filt).detach().numpy()

        
    probabilities[ifilter] = probabilities_filt
    probabilities_noise[ifilter] = probabilities_noise_filt
pkl.dump([models, probabilities, probabilities_noise], open("pickles/model_on_data.pkl","wb"))

Found 30 Models


100%|█████████████████████████████████████████████| 30/30 [00:00<00:00, 56.78it/s]


Found 30 Models


100%|█████████████████████████████████████████████| 30/30 [00:00<00:00, 67.04it/s]


Found 30 Models


100%|█████████████████████████████████████████████| 30/30 [00:00<00:00, 71.23it/s]
